# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
import os

# Colab Secrets bevorzugen; falls nicht vorhanden, auf Umgebungsvariablen zurückfallen
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    HUGGINGFACE_TOKEN = userdata.get("HUGGINGFACE_TOKEN")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_TOKEN")

# 👉 Cleanup (sehr wichtig)
if OPENAI_API_KEY:
    OPENAI_API_KEY = OPENAI_API_KEY.strip()

if HUGGINGFACE_TOKEN:
    HUGGINGFACE_TOKEN = HUGGINGFACE_TOKEN.strip()

# In Environment setzen (für Libraries nötig)
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

if HUGGINGFACE_TOKEN:
    os.environ["HUGGINGFACE_TOKEN"] = HUGGINGFACE_TOKEN

# Fehler, wenn Key fehlt
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY fehlt. Lege ihn in Colab unter Secrets an oder setze ihn per os.environ."
    )

In [5]:
# Nur die für dieses Notebook nötigen Pakete installieren.
# Fix pandas and requests versions to be compatible with google-colab.
# Colab brings a suitable numpy, avoid downgrading it.
%pip -q install -U \
  "langchain==0.2.16" \
  "langchain-community==0.2.16" \
  "langchain-openai==0.1.25" \
  "langchain-core==0.2.43" \
  "openai>=1.40,<2" \
  tiktoken \
  "pandas==2.2.2" \
  "requests==2.32.4"

In [6]:
from pathlib import Path
import pandas as pd

possible_paths = [
    Path("/content/Data (1).csv"),
    Path("/content/Data.csv"),
    Path("Data (1).csv"),
    Path("Data.csv"),
    Path("/mnt/data/Data (1).csv"),
]

for path in possible_paths:
    if path.exists():
        data_path = path
        break
else:
    raise FileNotFoundError(
        "Datensatz nicht gefunden. Erwarte z. B. '/content/Data (1).csv'."
    )

df = pd.read_csv(data_path)
print(f"Geladene Datei: {data_path}")
print(f"Shape: {df.shape}")

Geladene Datei: /content/Data (1).csv
Shape: (7, 2)


In [7]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [8]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains import LLMChain, SimpleSequentialChain, SequentialChain

In [9]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [10]:
# Niedrige Temperatur für stabile, reproduzierbare Antworten
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [11]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

In [12]:

chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipykernel_5455/546483037.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [13]:
product = "Queen Size Sheet Set"
chain.run(product)

/tmp/ipykernel_5455/550859008.py:2: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  chain.run(product)


"Choosing a name for a company that specializes in queen size sheet sets can be both fun and strategic. Here are some suggestions that convey comfort, quality, and the specific product focus:\n\n1. **Queen's Comfort**\n2. **Regal Rest**\n3. **Sheet Serenity**\n4. **Queenly Dreams**\n5. **Majestic Sheets**\n6. **Royal Slumber**\n7. **Dreamy Queen Sets**\n8. **Sovereign Sheets**\n9. **Noble Nesting**\n10. **Queen Size Haven**\n\nWhen selecting a name, consider your target audience, brand values, and the overall image you want to project. Make sure to check for domain availability if you plan to have an online presence!"

In [44]:
product = "Premium Organic Coffee Beans"
chain.run(product)



> Entering new MultiPromptChain chain...
None: {'input': 'Premium Organic Coffee Beans'}

KeyboardInterrupt: 

In [45]:
product = "AI-powered Fitness Coaching App"
chain.run(product)



> Entering new MultiPromptChain chain...
None: {'input': 'AI-powered Fitness Coaching App'}
> Finished chain.


"Creating an AI-powered fitness coaching app can be an exciting venture, combining technology with health and wellness. Here’s a comprehensive outline of features, functionalities, and considerations for developing such an app:\n\n### Key Features\n\n1. **Personalized Workout Plans**\n   - AI algorithms to create customized workout plans based on user goals (weight loss, muscle gain, endurance, etc.), fitness level, and preferences.\n   - Adaptive plans that evolve based on user progress and feedback.\n\n2. **Nutrition Guidance**\n   - Meal planning features that suggest recipes and meal prep ideas based on dietary preferences (vegan, keto, etc.) and caloric needs.\n   - Integration with food databases to track nutritional intake and provide feedback.\n\n3. **Progress Tracking**\n   - Tools for users to log workouts, track weight, and monitor body measurements.\n   - Visual progress reports and analytics to motivate users.\n\n4. **Virtual Coaching**\n   - AI-driven chatbots for real-ti

## SimpleSequentialChain

In [39]:
from langchain.chains import SimpleSequentialChain

In [40]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [41]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20-word description for the following company name: {text}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [42]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [46]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Choosing a name for your AI-powered fitness coaching app can be essential for branding and marketing. Here are some suggestions that convey the essence of fitness and AI:

1. **Fitelligence**
2. **AI Fit Coach**
3. **SmartFit**
4. **FitMentor**
5. **TrainAI**
6. **FitnessIQ**
7. **FitGenie**
8. **AI Gym Buddy**
9. **FitFusion**
10. **CoachBot**
11. **EmpowerFit**
12. **ActiveMind AI**
13. **NexGen Fitness**
14. **FitWise AI**
15. **Athlete AI Coach**

Make sure to check the availability of the name as a domain and on social media platforms to ensure a consistent online presence!
Fitelligence is an innovative AI-powered fitness coaching app, personalizing workouts and nutrition plans to empower users in achieving their fitness goals.

> Finished chain.


'Fitelligence is an innovative AI-powered fitness coaching app, personalizing workouts and nutrition plans to empower users in achieving their fitness goals.'

**Repeat the above twice for different products**

## SequentialChain

In [47]:
from langchain.chains import SequentialChain

In [48]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to English:\n\n{Review}"
)

chain_one = LLMChain(
    llm=llm,
    prompt=first_prompt,
    output_key="English_Review"
)

In [49]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)

chain_two = LLMChain(
    llm=llm,
    prompt=second_prompt,
    output_key="summary"
)

In [50]:
# prompt template 3: detect the source language of the review
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review written in? Answer with only the language.\n\n{Review}"
)

# chain 3: input=Review and output=language
chain_three = LLMChain(
    llm=llm,
    prompt=third_prompt,
    output_key="language"
)

In [51]:
# prompt template 4: follow-up message in the original review language
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a short follow-up response to the customer in {language}. "
    "Use this summary of the review: {summary}"
)
chain_four = LLMChain(
    llm=llm,
    prompt=fourth_prompt,
    output_key="followup_message"
)

In [52]:
# overall_chain: input=Review
# and output=English_Review, summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary", "followup_message"],
    verbose=True
)

In [53]:
review = df.Review[5]
overall_chain.invoke({"Review": review})



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': '"I find the taste mediocre. The foam doesn\'t hold, it\'s strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?"',
 'summary': 'The reviewer finds the taste mediocre, the foam unsatisfactory, and suspects the product could be an old batch or counterfeit compared to better versions bought in stores.',
 'followup_message': "Bonjour,\n\nMerci d'avoir pris le temps de partager votre avis. Nous sommes désolés d'apprendre que vous trouvez le goût et la mousse médiocres, et que vous suspectez un problème de qualité. Votre retour est précieux et nous allons l'examiner attentivement. N'hésitez pas à nous contacter directement pour que nous puissions résoudre ce souci ensemble.\n\nCordialement,  \n[Votre Nom]  \n[Votre Entreprise]"}

**Repeat the above twice for different products or reviews**

## Router Chain

In [54]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts,
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity.

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [55]:
prompt_infos = [
    {
        "name": "physics",
        "description": "Good for answering questions about physics",
        "prompt_template": physics_template
    },
    {
        "name": "math",
        "description": "Good for answering math questions",
        "prompt_template": math_template
    },
    {
        "name": "History",
        "description": "Good for answering history questions",
        "prompt_template": history_template
    },
    {
        "name": "computer science",
        "description": "Good for answering computer science questions",
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [56]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain, RouterOutputParser
from langchain.prompts import PromptTemplate

In [57]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [58]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain

destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [59]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [60]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string  // name of the prompt to use or "DEFAULT"
    "next_inputs": string  // a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [61]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [62]:
chain = MultiPromptChain(router_chain=router_chain,
                         destination_chains=destination_chains,
                         default_chain=default_chain, verbose=True
                        )

In [63]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


'Black body radiation refers to the electromagnetic radiation emitted by an idealized object known as a "black body," which absorbs all incident radiation, regardless of frequency or angle. A perfect black body is a theoretical concept that does not reflect or transmit any light, making it appear completely black at room temperature.\n\nThe key characteristics of black body radiation include:\n\n1. **Temperature Dependence**: The amount and spectrum of radiation emitted by a black body depend solely on its temperature. As the temperature increases, the intensity of the emitted radiation increases, and the peak wavelength of the radiation shifts to shorter wavelengths (this is described by Wien\'s displacement law).\n\n2. **Planck\'s Law**: Max Planck derived a formula that describes the spectral distribution of black body radiation. It shows that the energy emitted at a given wavelength is quantized, leading to the concept of quantized energy levels in quantum mechanics.\n\n3. **Stefan

In [64]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'To break down the problem \\(2 + 2\\):\n\n1. Identify the numbers involved: We have the number 2 and another number 2.\n2. Understand the operation: The operation here is addition, which means we are combining the two numbers.\n\nNow, we can perform the addition:\n\n\\[\n2 + 2 = 4\n\\]\n\nSo, the answer to the question \\(2 + 2\\) is \\(4\\).'

In [65]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
biology: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA serves as the fundamental blueprint for life. Here are several key reasons why this is the case:\n\n1. **Genetic Information Storage**: DNA (deoxyribonucleic acid) contains the genetic instructions necessary for the development, functioning, growth, and reproduction of all living organisms. It encodes the information required to produce proteins, which perform a vast array of functions within the cell.\n\n2. **Cellular Function and Identity**: Each cell in our body has a specific role, whether it be a muscle cell, nerve cell, or skin cell. The DNA in each cell contains the complete set of instructions for the organism, but different cells express different genes. This selective gene expression allows cells to specialize and perform their unique functions while still retaining the complete genetic information.\n\n3. **Replication and Inheritance**: DNA is capable of self-replication, which is essential for cell division. When a cell divid

**Repeat the above at least once for different inputs and chains executions - Be creative!**